In [ ]:
import numpy as np
import pandas as pd

df = pd.read_parquet('../data/silver/eventos_challenger_league_adjusted.parquet')

df['progression_x_m'] = df['x_destino_m'] - df['x_origen_m']

df['distancia_m'] = np.sqrt((df['x_destino_m'] - df['x_origen_m'])**2 + (df['y_destino_m'] - df['y_origen_m'])**2)

xt_matrix = np.array([
    [0.00638, 0.00779, 0.00844, 0.00927, 0.01126, 0.01248, 0.01473, 0.01853, 0.02412, 0.02755, 0.03467, 0.03792],
    [0.00739, 0.00896, 0.00977, 0.01070, 0.01275, 0.01400, 0.01683, 0.02122, 0.02756, 0.03178, 0.05812, 0.04617],
    [0.00846, 0.01022, 0.01115, 0.01214, 0.01423, 0.01549, 0.01852, 0.02412, 0.03046, 0.03608, 0.09139, 0.06388],
    [0.00958, 0.01132, 0.01228, 0.01322, 0.01511, 0.01633, 0.01948, 0.02534, 0.03185, 0.03819, 0.11186, 0.07436],
    [0.00958, 0.01132, 0.01228, 0.01322, 0.01511, 0.01633, 0.01948, 0.02534, 0.03185, 0.03819, 0.11186, 0.07436],
    [0.00846, 0.01022, 0.01115, 0.01214, 0.01423, 0.01549, 0.01852, 0.02412, 0.03046, 0.03608, 0.09139, 0.06388],
    [0.00739, 0.00896, 0.00977, 0.01070, 0.01275, 0.01400, 0.01683, 0.02122, 0.02756, 0.03178, 0.05812, 0.04617],
    [0.00638, 0.00779, 0.00844, 0.00927, 0.01126, 0.01248, 0.01473, 0.01853, 0.02412, 0.02755, 0.03467, 0.03792]
])



def extraer_xt(y, x):
    if pd.isna(y) or pd.isna(x):
        return 0.0
    idx_y, idx_x = int(y), int(x)
    if not (0 <= idx_y < 8 and 0 <= idx_x < 12):
        return 0.0
    return xt_matrix[idx_y, idx_x]

def evaluar_accion(row):
    
    # 1. passes
    if row['event_category'] == 'passes':
        # Tus variables limpias
        exitoso = row['outcome'] == True
        xt_origen = row['xT_origen']
        xt_destino = row['xT_destino']
        is_cross = row['eventActionType'] == "cross"
        is_long_pass = row['isLongBall'] == True
        
        if not exitoso or pd.isna(xt_destino):
            coste_base = -xt_origen
            if is_long_pass or (pd.notna(row['progression_x_m']) and row['progression_x_m'] > 20):
                return coste_base * 0.70
            else:
                return coste_base
        else:
            xt_bruto = xt_destino - xt_origen
            if xt_bruto <= 0: return xt_bruto
            if is_cross: return xt_bruto * 0.75
            if is_long_pass and (pd.notna(row['progression_x_m']) and row['progression_x_m'] > 25):
                return xt_bruto * 1.15
            return xt_bruto

    # 2. carries (Al espacio, sin presión)
    elif row['eventActionType'] == 'ball-carry':
        xt_bruto = row['xT_destino'] - row['xT_origen']
        # Si la progresión es negativa (hacia atrás), devolvemos 0 o el valor negativo
        return xt_bruto

    # 3. dribbleS (1 vs 1)
    elif row['eventActionType'] == 'dribble':
        exitoso = row['outcome'] == True
        # El dribble es en una baldosa. Recompensamos superar al rival (ej: +20% del valor de la zona)
        # o castigamos la pérdida del balón en esa zona.
        if exitoso:
            return row['xT_origen'] * 0.20 
        else:
            return -row['xT_origen']

    # 4. RESTO DE EVENTOS (Defensa, Tiros, Recuperaciones)
    elif row['event_category'] == 'defensive':
        tipo = row['eventActionType']
        exitoso = row['outcome'] == True
        xt_origen = row['xT_origen']

        if tipo == 'interception':
            return 0.50 * xt_origen
        elif tipo == 'ball-recovery':
            return 0.35 * xt_origen
        elif tipo == 'tackle':
            if exitoso:
                return 0.25*xt_origen
            else:
                return -xt_origen * 0.25
        elif tipo in ['clearance', 'block']:
            return 0.0
    return 0.0

df['xT_origen'] = df.apply(lambda row: extraer_xt(row['xt_zona_y_origen'], row['xt_zona_x_origen']), axis=1)
df['xT_destino'] = df.apply(lambda row: extraer_xt(row['xt_zona_y_destino'], row['xt_zona_x_destino']), axis=1)
df['xT_delta'] = df['xT_destino'] - df['xT_origen']

df['xT_delta_adjusted'] = df.apply(evaluar_accion, axis=1)
df.groupby(['event_category', 'eventActionType'])['xT_delta_adjusted'].describe()
df.to_parquet('../data/gold/challenger_pro_league_xT.parquet', index=False)

In [2]:
import pandas as pd
lineups = pd.read_parquet('../data/bronze/lineups.parquet')
print("Columnas de lineups:", lineups.columns.tolist())
print(lineups.head(2))

Columnas de lineups: ['round', 'match_id', 'team_side', 'player_id', 'player_name', 'position', 'is_starter', 'minutes_played']
   round  match_id team_side  player_id      player_name position  is_starter  \
0      1  16384051      home    1155612  Leonardo Cascio        G        True   
1      1  16384051      home     901897     Loic Masscho        D        True   

   minutes_played  
0              90  
1              90  


In [ ]:
import numpy as np
import pandas as pd

df_events = pd.read_parquet('../data/gold/challenger_pro_league_xT.parquet')
df_lineups = pd.read_parquet('../data/bronze/lineups.parquet')

#Posicion más frecuente de cada jugador, ya que puede haber alternado
frequent_position = (
    df_lineups.groupby('player_id')['position']
    .agg(lambda s: s.dropna().mode().iloc[0] if not s.empty else 'Unknown')
    .to_dict()
)

df_profiles = df_lineups.groupby('player_id', as_index=False).agg(
    player_name=('player_name', 'last'),
    matches_called=('match_id', 'nunique'),
    starter=('is_starter', 'sum'),
    total_minutes=('minutes_played', 'sum')
)
df_profiles['position'] = df_profiles['player_id'].map(frequent_position)

#Vectorizamos las columnas de xT para cada tipo de acción
df_events['xt_pass'] = np.where(df_events['event_category'] == 'passes', df_events['xT_delta_ajustado'], 0.0)
df_events['xt_carry'] = np.where(df_events['eventActionType'] == 'ball-carry', df_events['xT_delta_ajustado'], 0.0)
df_events['xt_dribble'] = np.where(df_events['eventActionType'] == 'dribble', df_events['xT_delta_ajustado'], 0.0)
df_events['xt_defensive'] = np.where(df_events['event_category'] == 'defensive', df_events['xT_delta_ajustado'], 0.0)

df_events['count_pass'] = (df_events['event_category'] == 'passes').astype(int)
df_events['count_carry'] = (df_events['eventActionType'] == 'ball-carry').astype(int)
df_events['count_dribble'] = (df_events['eventActionType'] == 'dribble').astype(int)
df_events['count_defensive'] = (df_events['event_category'] == 'defensive').astype(int)

df_xt_totales = df_events.groupby('player_id', as_index=False).agg(
    # Amenaza acumulada
    xt_pass_total=('xt_pass', 'sum'),
    xt_carry_total=('xt_carry', 'sum'),
    xt_dribble_total=('xt_dribble', 'sum'),
    xt_defensive_total=('xt_defensive', 'sum'),
    xt_total=('xT_delta_ajustado', 'sum'),
    # Volumen acumulado
    passes_total=('count_pass', 'sum'),
    carries_total=('count_carry', 'sum'),
    dribbles_total=('count_dribble', 'sum'),
    defensive_total=('count_defensive', 'sum')
)

df_gold = pd.merge(df_profiles, df_xt_totales, on='player_id', how='left').fillna(0.0)

#Se divid
factor_90 = np.where(df_gold['total_minutes'] > 0, 90.0 / df_gold['total_minutes'], 0.0)

columns_to_normalize = [
    'xt_pass_total', 'xt_carry_total', 'xt_dribble_total', 
    'xt_defensive_total', 'xt_total',
    'passes_total', 'carries_total', 'dribbles_total', 'defensive_total'
]

for col in columns_to_normalize:
    col_p90 = col.replace('_total', '') + '_p90'
    df_gold[col_p90] = df_gold[col] * factor_90

MINUTES_CUT = 90
df_gold_filtered = df_gold[
    (df_gold['total_minutes'] >= MINUTES_CUT) & 
    (df_gold['position'] != 'G')
].sort_values(by='xt_p90', ascending=False).reset_index(drop=True)

# 8. Persistencia en Capa Oro
df_gold_filtered.to_parquet('../data/gold/scouting_xt_profiles_gold.parquet', index=False)

df_gold_filtered[['player_name', 'position', 'total_minutes', 'xt_pass_p90', 'xt_carry_p90', 'xt_dribble_p90', 'xt_defensive_p90']].head()

,player_name,position,total_minutes,xt_pass_p90,xt_carry_p90,xt_dribble_p90,xt_defensive_p90
0,Milan Govaers,M,156,0.333459,0.089117,0.000000,0.009642
1,Ruslan Vydysh,F,152,-0.010743,0.186253,0.008458,0.043305
2,Axl Wins,D,90,0.089241,0.085570,0.010136,0.023355
3,Nunzio Engwanda,D,270,0.183417,0.012520,-0.017599,0.012509
4,Bo De Kerf,D,276,0.022729,0.108098,-0.004560,0.043152


In [ ]:
import duckdb
import time

con = duckdb.connect()
print("Iniciando procesamiento de Plata a Oro para Tiros")
inicio = time.time()

query_shots_gold = """
COPY (
    WITH base_lineups AS (
        SELECT 
            player_id,
            LAST(player_name) as player_name,
            MODE(position) as position,
            COUNT(DISTINCT match_id) as matches_called,
            SUM(CASE WHEN is_starter THEN 1 ELSE 0 END) as starts,
            SUM(minutes_played) as total_minutes
        FROM read_parquet('../data/bronze/lineups.parquet')
        GROUP BY player_id
    ),

    base_shots AS (
        SELECT 
            player_id,
            COUNT(*) as shots_total,
            SUM(CASE WHEN shot_type = 'goal' THEN 1 ELSE 0 END) AS goals_total,
            SUM(xg) AS xg_total,
            SUM(xgot) AS xgot_total,

            --FINISHING SKILL TOTAL: Diferencia entre goles y xG, para medir la capacidad de definición de un jugador
            SUM(CASE WHEN shot_type = 'goal' THEN 1.0 ELSE 0.0 END) - SUM(xg) AS finishing_skill_total,

            -- Shooting Added Value: xGOT - xG (excluyendo tiros bloqueados)
            SUM(CASE WHEN shot_type != 'block' THEN (xgot - xg) ELSE 0.0 END) AS sav_total,
            AVG(CASE WHEN shot_type != 'block' THEN (xgot - xg) ELSE NULL END) AS sav_per_shot
        FROM read_parquet('../data/silver/shots_silver.parquet')
        GROUP BY player_id
    ),

    joined_profiles AS (
        -- Cruce relacional: mantenemos a todos los jugadores con minutos
        SELECT 
            l.player_id,
            l.player_name,
            l.position,
            l.matches_called,
            l.starts,
            l.total_minutes,
            
            -- Imputación de ceros en jugadores sin disparos registrados
            COALESCE(s.shots_total, 0) AS shots_total,
            COALESCE(s.goals_total, 0) AS goals_total,
            COALESCE(s.xg_total, 0.0) AS xg_total,
            COALESCE(s.xgot_total, 0.0) AS xgot_total,
            COALESCE(s.finishing_skill_total, 0.0) AS finishing_skill_total,
            COALESCE(s.sav_total, 0.0) AS sav_total,
            COALESCE(s.sav_per_shot, 0.0) AS sav_per_shot
        FROM base_lineups l
        LEFT JOIN base_shots s ON l.player_id = s.player_id
    )

    SELECT 
        player_id,
        player_name,
        position,
        matches_called,
        starts,
        total_minutes,
        shots_total,
        goals_total,
        xg_total,
        xgot_total,
        finishing_skill_total,
        sav_total,
        sav_per_shot,
        
        -- Normalización por 90 minutos
        ROUND((shots_total / total_minutes) * 90.0, 3) AS shots_p90,
        ROUND((goals_total / total_minutes) * 90.0, 3) AS goals_p90,
        ROUND((xg_total / total_minutes) * 90.0, 3) AS xg_p90,
        ROUND((xgot_total / total_minutes) * 90.0, 3) AS xgot_p90,
        ROUND((finishing_skill_total / total_minutes) * 90.0, 3) AS finishing_skill_p90,
        ROUND((sav_total / total_minutes) * 90.0, 3) AS sav_p90
        
    FROM joined_profiles
    WHERE total_minutes >= 90 
      AND position != 'G'
    ORDER BY finishing_skill_total DESC
) TO '../data/gold/challenger_pro_league_xG.parquet' (FORMAT PARQUET);
"""

con.execute(query_shots_gold)

fin = time.time()
print(f"¡Capa Gold de disparos generada en {fin - inicio:.2f} segundos!")

df_top_finishers = con.execute("""
    SELECT 
        player_name, 
        position, 
        total_minutes, 
        goals_total, 
        ROUND(xg_total, 2) AS xg, 
        ROUND(finishing_skill_total, 2) AS fin_skill,
        ROUND(sav_per_shot, 3) AS sav_shot,
        goals_p90,
        xg_p90
    FROM '../data/gold/challenger_pro_league_xG.parquet'
    WHERE shots_total >= 3
    ORDER BY fin_skill DESC
    LIMIT 5
""").df()

print(df_top_finishers)

Iniciando procesamiento de Plata a Oro para Tiros
¡Capa Gold de disparos generada en 0.11 segundos!
       player_name position  total_minutes  goals_total    xg  fin_skill  \
0       Bo De Kerf        D          276.0          2.0  0.29       1.71   
1      Arnold Vula        F          236.0          3.0  1.41       1.59   
2     Naïm Amengai        M          180.0          2.0  0.56       1.44   
3  Lukas Van Eenoo        M          347.0          2.0  0.57       1.43   
4     Simon Paulet        D          178.0          3.0  1.72       1.28   

   sav_shot  goals_p90  xg_p90  
0     0.239      0.652   0.095  
1     0.256      1.144   0.539  
2     0.134      1.000   0.282  
3     0.198      0.519   0.148  
4     0.066      1.517   0.869  


In [ ]:
import duckdb
import os

con = duckdb.connect()

silver_dir = "../data/silver"
gold_dir = "../data/gold"
os.makedirs(gold_dir, exist_ok=True)

query_player_match_gold = f"""
COPY (
    WITH spine AS (
        SELECT 
            round,
            match_id,
            team_side,
            team_name,
            player_id,
            player_name,
            position,
            is_starter,
            minutes_played
        FROM read_parquet('{silver_dir}/lineups_silver.parquet')
    ),
    
    eventos_partido AS (
        -- Agregación exhaustiva de acciones con balón y sin balón
        SELECT 
            match_id,
            player_id,

            --xT
            ROUND(SUM(xT_delta_ajustado), 4) AS xt_total,
            ROUND(SUM(CASE WHEN event_category = 'passes' THEN xT_delta_ajustado ELSE 0 END), 4) AS xt_passes,
            ROUND(SUM(CASE WHEN event_category = 'ball-carries' THEN xT_delta_ajustado ELSE 0 END), 4) AS xt_carries,

            -- 
            COUNT(*) FILTER (WHERE event_category = 'passes') AS passes_total,
            COUNT(*) FILTER (WHERE event_category = 'passes' AND outcome = true) AS passes_completed,
            COUNT(*) FILTER (WHERE eventActionType = 'cross') AS crosses_total,
            COUNT(*) FILTER (WHERE eventActionType = 'cross' AND outcome = true) AS crosses_completed,

            -- Movimiento hacia adelante
            COUNT(*) FILTER (WHERE keypass = true) AS key_passes,
            COUNT(*) FILTER (WHERE isAssist = true) AS assists_total,
            COUNT(*) FILTER (WHERE event_category = 'passes' AND progression_x_m >= 9.14) AS progressive_passes,
            COUNT(*) FILTER (WHERE event_category = 'ball-carries' AND progression_x_m >= 9.14) AS progressive_carries,

            -- Regates
            COUNT(*) FILTER (WHERE eventActionType = 'dribble') AS dribbles_total,
            COUNT(*) FILTER (WHERE eventActionType = 'dribble' AND outcome = true) AS dribbles_completed,

            -- Defensa
            COUNT(*) FILTER (WHERE eventActionType = 'ball-recovery') AS recoveries,
            COUNT(*) FILTER (WHERE eventActionType = 'interception') AS interceptions,
            COUNT(*) FILTER (WHERE eventActionType = 'tackle') AS tackles_total,
            COUNT(*) FILTER (WHERE eventActionType = 'tackle' AND outcome = true) AS tackles_won,
            COUNT(*) FILTER (WHERE eventActionType = 'clearance') AS clearances,
            COUNT(*) FILTER (WHERE eventActionType = 'block') AS blocks

        FROM read_parquet('../data/gold/challenger_pro_league_xT.parquet')
        GROUP BY match_id, player_id
    ),
    
    xg_partido AS (
        -- Agregación de volumen de finalización por partido desde Silver
        SELECT 
            match_id,
            player_id,
            COUNT(*) AS shots_total,
            COUNT(*) FILTER (WHERE shot_type = 'goal') AS goals_total,
            ROUND(SUM(xg), 4) AS xg_total,
            ROUND(SUM(xgot), 4) AS xgot_total,
            ROUND(SUM(xgot - xg), 4) AS sav_total
        FROM read_parquet('{silver_dir}/shots_silver.parquet')
        GROUP BY match_id, player_id
    )

    -- Unión dimensional: Spine + Métricas imputadas a cero
    SELECT 
        s.round,
        s.match_id,
        s.team_name,
        s.team_side,
        s.player_id,
        s.player_name,
        s.position,
        s.is_starter,
        s.minutes_played,
        
        -- Progresión territorial
        COALESCE(e.xt_total, 0.0) AS xt_total,
        COALESCE(e.xt_passes, 0.0) AS xt_passes,
        COALESCE(e.xt_carries, 0.0) AS xt_carries,

        -- Distribución
        COALESCE(e.passes_total, 0) AS passes_total,
        COALESCE(e.passes_completed, 0) AS passes_completed,
        COALESCE(e.crosses_total, 0) AS crosses_total,
        COALESCE(e.crosses_completed, 0) AS crosses_completed,

        -- Creación y verticalidad
        COALESCE(e.key_passes, 0) AS key_passes,
        COALESCE(e.assists_total, 0) AS assists_total,
        COALESCE(e.progressive_passes, 0) AS progressive_passes,
        COALESCE(e.progressive_carries, 0) AS progressive_carries,

        -- Desborde
        COALESCE(e.dribbles_total, 0) AS dribbles_total,
        COALESCE(e.dribbles_completed, 0) AS dribbles_completed,

        -- Trabajo defensivo
        COALESCE(e.recoveries, 0) AS recoveries,
        COALESCE(e.interceptions, 0) AS interceptions,
        COALESCE(e.tackles_total, 0) AS tackles_total,
        COALESCE(e.tackles_won, 0) AS tackles_won,
        COALESCE(e.clearances, 0) AS clearances,
        COALESCE(e.blocks, 0) AS blocks,

        -- Remate
        COALESCE(x.shots_total, 0) AS shots_total,
        COALESCE(x.goals_total, 0) AS goals_total,
        COALESCE(x.xg_total, 0.0) AS xg_total,
        COALESCE(x.xgot_total, 0.0) AS xgot_total,
        COALESCE(x.sav_total, 0.0) AS sav_total

    FROM spine s
    LEFT JOIN eventos_partido e 
        ON s.match_id = e.match_id AND s.player_id = e.player_id
    LEFT JOIN xg_partido x 
        ON s.match_id = x.match_id AND s.player_id = x.player_id
) TO '{gold_dir}/player_match_gold.parquet' (FORMAT PARQUET);
"""

con.execute(query_player_match_gold)
print("Muestra de Player Match Gold:")
con.execute("""
    SELECT 
        player_name, 
        position, 
        minutes_played, 
        recoveries, 
        interceptions, 
        tackles_won, 
        clearances,
        progressive_passes
    FROM '../data/gold/player_match_gold.parquet'
    WHERE team_name = 'Koninklijke Sporting Hasselt' AND minutes_played > 45
    ORDER BY clearances DESC, recoveries DESC
    LIMIT 5
""").df()


Muestra de Player Match Gold:


,player_name,position,minutes_played,recoveries,interceptions,tackles_won,clearances,progressive_passes
0,Cheick Thiam,D,90,3,0,1,10,17
1,Andres Labie,D,90,2,3,1,10,21
2,Andres Labie,D,90,2,1,2,7,19
3,Andres Labie,D,90,6,2,1,5,22
4,Nicolas Gerits,M,90,6,1,2,4,18


In [ ]:
import duckdb

con = duckdb.connect()

query_master_gold = """
COPY (
    WITH raw_agg AS (
        SELECT 
            player_id,
            player_name,
            team_name,
            position,
            SUM(minutes_played) AS total_minutes,
            COUNT(DISTINCT match_id) AS matches_played,
            
            -- Volúmenes brutos: xT
            ROUND(SUM(xt_total), 4) AS xt_total,
            ROUND(SUM(xt_passes), 4) AS xt_passes,
            ROUND(SUM(xt_carries), 4) AS xt_carries,
            SUM(passes_total) AS passes_total,
            SUM(passes_completed) AS passes_completed,
            
            -- Volúmenes brutos: Finalización
            SUM(shots_total) AS shots_total,
            SUM(goals_total) AS goals_total,
            ROUND(SUM(xg_total), 4) AS xg_total,
            ROUND(SUM(xgot_total), 4) AS xgot_total,
            ROUND(SUM(sav_total), 4) AS sav_total
            
        FROM read_parquet('../data/gold/player_match_gold.parquet')
        GROUP BY player_id, player_name, team_name, position
    )
    SELECT 
        *,
        -- Ratios de eficiencia
        ROUND((passes_completed * 100.0) / NULLIF(passes_total, 0), 2) AS pass_accuracy_pct,
        ROUND(sav_total / NULLIF(shots_total, 0), 4) AS sav_per_shot,
        
        -- Métricas normalizadas P90 (La base para los percentiles)
        ROUND((xt_total / total_minutes) * 90, 4) AS xt_p90,
        ROUND((xt_passes / total_minutes) * 90, 4) AS xt_passes_p90,
        ROUND((xt_carries / total_minutes) * 90, 4) AS xt_carries_p90,
        ROUND((passes_total * 1.0 / total_minutes) * 90, 2) AS passes_p90,
        ROUND((shots_total * 1.0 / total_minutes) * 90, 2) AS shots_p90,
        ROUND((goals_total * 1.0 / total_minutes) * 90, 2) AS goals_p90,
        ROUND((xg_total / total_minutes) * 90, 4) AS xg_p90,
        ROUND((xgot_total / total_minutes) * 90, 4) AS xgot_p90,
        ROUND((sav_total / total_minutes) * 90, 4) AS sav_p90

    FROM raw_agg
    ORDER BY total_minutes DESC
) TO '../data/gold/master_player_gold.parquet' (FORMAT PARQUET);
"""

con.execute(query_master_gold)

query_percentiles = """
COPY (
    WITH filtrados AS (
        SELECT *
        FROM read_parquet('../data/gold/master_player_gold.parquet')
        WHERE total_minutes >= 180
    ),
    
    percentiles_generales AS (
        SELECT 
            *,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY xt_p90) * 100, 1) AS pct_xt,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY passes_p90) * 100, 1) AS pct_passes_vol,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY pass_accuracy_pct) * 100, 1) AS pct_pass_acc,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY shots_p90) * 100, 1) AS pct_shots,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY xg_p90) * 100, 1) AS pct_xg
        FROM filtrados
    ),
    
    percentiles_tiro AS (
        -- Solo jugadores con tiros a puerta
        SELECT 
            player_id,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY sav_per_shot) * 100, 1) AS pct_sav_quality
        FROM filtrados
        WHERE shots_total > 0
    )

    SELECT 
        g.player_id,
        g.player_name,
        g.team_name,
        g.position,
        g.total_minutes,
        g.matches_played,
        g.xt_p90,
        g.passes_p90,
        g.pass_accuracy_pct,
        g.shots_p90,
        g.xg_p90,
        g.sav_per_shot,
        g.pct_xt,
        g.pct_passes_vol,
        g.pct_pass_acc,
        g.pct_shots,
        g.pct_xg,
        t.pct_sav_quality
    FROM percentiles_generales g
    LEFT JOIN percentiles_tiro t ON g.player_id = t.player_id
    ORDER BY g.position, g.pct_xt DESC
) TO '../data/gold/scouting_percentiles_gold.parquet' (FORMAT PARQUET);
"""

con.execute(query_percentiles)

con.execute("""
    SELECT 
        player_name, 
        position, 
        total_minutes, 
        pct_xt, 
        pct_xg, 
        pct_sav_quality
    FROM '../data/gold/scouting_percentiles_gold.parquet'
    WHERE team_name = 'Koninklijke Sporting Hasselt'
    ORDER BY total_minutes DESC
""").df()


,player_name,position,total_minutes,pct_xt,pct_xg,pct_sav_quality
0,Cheick Thiam,D,270.0,78.0,0.0,NaN
1,Andres Labie,D,270.0,64.0,0.0,NaN
2,Leonardo Cascio,G,270.0,35.7,0.0,NaN
3,Mehdi Bounou,F,269.0,23.5,5.9,70.6
4,Lucas Vankerkhoven,M,265.0,25.6,0.0,NaN
5,Lucca Lucker,D,256.0,24.0,0.0,NaN
6,Loic Masscho,D,253.0,2.0,0.0,NaN
7,Siebe Van Keymolen,F,250.0,35.3,76.5,29.4
8,Nicolas Gerits,M,232.0,53.8,38.5,23.5
9,Livio Milts,M,229.0,12.8,82.1,55.9


In [15]:
import duckdb

con = duckdb.connect()

query_master_gold = """
COPY (
    WITH raw_agg AS (
        SELECT 
            player_id,
            player_name,
            team_name,
            position,
            
            -- Métricas de rol y participación
            SUM(minutes_played) AS total_minutes,
            COUNT(DISTINCT match_id) AS matches_played,
            COUNT(*) FILTER (WHERE is_starter = true) AS matches_started,
            COUNT(*) FILTER (WHERE is_starter = false AND minutes_played > 0) AS sub_appearances,
            ROUND(AVG(minutes_played), 1) AS avg_minutes_per_match,
            
            -- Volúmenes: Progresión (xT)
            ROUND(SUM(xt_total), 4) AS xt_total,
            ROUND(SUM(xt_passes), 4) AS xt_passes,
            ROUND(SUM(xt_carries), 4) AS xt_carries,
            SUM(progressive_passes) AS prog_passes_total,
            SUM(progressive_carries) AS prog_carries_total,
            
            -- Volúmenes: Pases y Creación
            SUM(passes_total) AS passes_total,
            SUM(passes_completed) AS passes_completed,
            SUM(crosses_total) AS crosses_total,
            SUM(crosses_completed) AS crosses_completed,
            SUM(key_passes) AS key_passes_total,
            SUM(assists_total) AS assists_total,
            
            -- Volúmenes: Desborde (1 vs 1)
            SUM(dribbles_total) AS dribbles_total,
            SUM(dribbles_completed) AS dribbles_completed,
            
            -- Volúmenes: Fase defensiva
            SUM(recoveries) AS recoveries_total,
            SUM(interceptions) AS interceptions_total,
            SUM(tackles_total) AS tackles_total,
            SUM(tackles_won) AS tackles_won_total,
            SUM(clearances) AS clearances_total,
            SUM(blocks) AS blocks_total,
            
            -- Volúmenes: Finalización
            SUM(shots_total) AS shots_total,
            SUM(goals_total) AS goals_total,
            ROUND(SUM(xg_total), 4) AS xg_total,
            ROUND(SUM(xgot_total), 4) AS xgot_total,
            ROUND(SUM(sav_total), 4) AS sav_total
            
        FROM read_parquet('../data/gold/player_match_gold.parquet')
        GROUP BY player_id, player_name, team_name, position
    )
    SELECT 
        *,
        -- Ratios de eficacia (%)
        ROUND((passes_completed * 100.0) / NULLIF(passes_total, 0), 1) AS pass_accuracy_pct,
        ROUND((crosses_completed * 100.0) / NULLIF(crosses_total, 0), 1) AS cross_accuracy_pct,
        ROUND((dribbles_completed * 100.0) / NULLIF(dribbles_total, 0), 1) AS dribble_success_pct,
        ROUND((tackles_won_total * 100.0) / NULLIF(tackles_total, 0), 1) AS tackle_success_pct,
        ROUND(sav_total / NULLIF(shots_total, 0), 4) AS sav_per_shot,
        
        -- Normalizaciones P90 (Métricas comparables)
        ROUND((xt_total / total_minutes) * 90, 4) AS xt_p90,
        ROUND((prog_passes_total * 1.0 / total_minutes) * 90, 2) AS prog_passes_p90,
        ROUND((prog_carries_total * 1.0 / total_minutes) * 90, 2) AS prog_carries_p90,
        ROUND((passes_total * 1.0 / total_minutes) * 90, 2) AS passes_p90,
        ROUND((key_passes_total * 1.0 / total_minutes) * 90, 2) AS key_passes_p90,
        ROUND((assists_total * 1.0 / total_minutes) * 90, 2) AS assists_p90,
        ROUND((dribbles_total * 1.0 / total_minutes) * 90, 2) AS dribbles_p90,
        ROUND((recoveries_total * 1.0 / total_minutes) * 90, 2) AS recoveries_p90,
        ROUND((interceptions_total * 1.0 / total_minutes) * 90, 2) AS interceptions_p90,
        ROUND((tackles_won_total * 1.0 / total_minutes) * 90, 2) AS tackles_won_p90,
        ROUND((clearances_total * 1.0 / total_minutes) * 90, 2) AS clearances_p90,
        ROUND((shots_total * 1.0 / total_minutes) * 90, 2) AS shots_p90,
        ROUND((xg_total / total_minutes) * 90, 4) AS xg_p90,
        ROUND((goals_total * 1.0 / total_minutes) * 90, 2) AS goals_p90

    FROM raw_agg
    ORDER BY total_minutes DESC
) TO '../data/gold/master_player_gold.parquet' (FORMAT PARQUET);
"""

con.execute(query_master_gold)

con.execute("""
    SELECT 
        player_name, 
        position, 
        matches_started, 
        sub_appearances, 
        total_minutes,
        recoveries_p90, 
        prog_passes_p90, 
        key_passes_p90, 
        xg_p90
    FROM '../data/gold/master_player_gold.parquet'
    WHERE team_name = 'Koninklijke Sporting Hasselt'
    ORDER BY total_minutes DESC
""").df()

query_scouting_percentiles = """
COPY (
    WITH filtrados AS (
        -- Filtro estricto de muestra representativa
        SELECT *
        FROM read_parquet('../data/gold/master_player_gold.parquet')
        WHERE total_minutes >= 180
    ),
    
    percentiles_generales AS (
        SELECT 
            *,
            -- Progresión y distribución
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY xt_p90) * 100, 1) AS pct_xt,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY prog_passes_p90) * 100, 1) AS pct_prog_passes,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY passes_p90) * 100, 1) AS pct_passes_vol,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY pass_accuracy_pct) * 100, 1) AS pct_pass_acc,
            
            -- Creación y desborde
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY key_passes_p90) * 100, 1) AS pct_key_passes,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY dribbles_p90) * 100, 1) AS pct_dribbles,
            
            -- Fase defensiva
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY recoveries_p90) * 100, 1) AS pct_recoveries,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY interceptions_p90) * 100, 1) AS pct_interceptions,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY tackles_won_p90) * 100, 1) AS pct_tackles_won,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY clearances_p90) * 100, 1) AS pct_clearances,
            
            -- Finalización (volumen)
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY shots_p90) * 100, 1) AS pct_shots,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY xg_p90) * 100, 1) AS pct_xg
        FROM filtrados
    ),
    
    percentiles_tiro AS (
        -- Aislamiento de calidad de golpeo para no penalizar a los no-tiradores
        SELECT 
            player_id,
            ROUND(PERCENT_RANK() OVER (PARTITION BY position ORDER BY sav_per_shot) * 100, 1) AS pct_sav_quality
        FROM filtrados
        WHERE shots_total > 0
    )

    SELECT 
        g.*,
        t.pct_sav_quality
    FROM percentiles_generales g
    LEFT JOIN percentiles_tiro t ON g.player_id = t.player_id
    ORDER BY g.position, g.pct_xt DESC
) TO '../data/gold/scouting_percentiles_gold.parquet' (FORMAT PARQUET);
"""

con.execute(query_scouting_percentiles)


con.execute("""
    SELECT 
        player_name, 
        position, 
        total_minutes, 
        pct_xt, 
        pct_xg, 
        pct_sav_quality
    FROM '../data/gold/scouting_percentiles_gold.parquet'
    WHERE team_name = 'Koninklijke Sporting Hasselt'
    ORDER BY total_minutes DESC
""").df()


,player_name,position,total_minutes,pct_xt,pct_xg,pct_sav_quality
0,Cheick Thiam,D,270.0,78.0,0.0,NaN
1,Andres Labie,D,270.0,64.0,0.0,NaN
2,Leonardo Cascio,G,270.0,35.7,0.0,NaN
3,Mehdi Bounou,F,269.0,23.5,5.9,70.6
4,Lucas Vankerkhoven,M,265.0,25.6,0.0,NaN
5,Lucca Lucker,D,256.0,24.0,0.0,NaN
6,Loic Masscho,D,253.0,2.0,0.0,NaN
7,Siebe Van Keymolen,F,250.0,35.3,76.5,29.4
8,Nicolas Gerits,M,232.0,53.8,38.5,23.5
9,Livio Milts,M,229.0,12.8,82.1,55.9


In [ ]:
import duckdb

con = duckdb.connect()

query_team_match_gold = """
COPY (
    WITH team_raw AS (
        SELECT
            round,
            match_id,
            team_name,
            team_side,
            
            -- Marcador y remate
            SUM(goals_total) AS goals_for,
            ROUND(SUM(xg_total), 4) AS xg_for,
            ROUND(SUM(xgot_total), 4) AS xgot_for,
            SUM(shots_total) AS shots_for,
            ROUND(SUM(sav_total), 4) AS sav_total,
            
            -- Progresión territorial (xT)
            ROUND(SUM(xt_total), 4) AS xt_total,
            ROUND(SUM(xt_passes), 4) AS xt_passes,
            ROUND(SUM(xt_carries), 4) AS xt_carries,
            
            -- Circulación y creación
            SUM(passes_total) AS passes_total,
            SUM(passes_completed) AS passes_completed,
            SUM(progressive_passes) AS prog_passes_total,
            SUM(progressive_carries) AS prog_carries_total,
            SUM(crosses_total) AS crosses_total,
            SUM(crosses_completed) AS crosses_completed,
            SUM(key_passes) AS key_passes_total,
            SUM(assists_total) AS assists_total,
            
            -- Desborde (1 vs 1)
            SUM(dribbles_total) AS dribbles_total,
            SUM(dribbles_completed) AS dribbles_completed,
            
            -- Fase defensiva colectiva
            SUM(recoveries) AS recoveries_total,
            SUM(interceptions) AS interceptions_total,
            SUM(tackles_total) AS tackles_total,
            SUM(tackles_won) AS tackles_won_total,
            SUM(clearances) AS clearances_total,
            SUM(blocks) AS blocks_total
        FROM read_parquet('../data/gold/player_match_gold.parquet')
        GROUP BY round, match_id, team_name, team_side
    )

    SELECT 
        t.round,
        t.match_id,
        t.team_name,
        t.team_side,
        opp.team_name AS opponent_name,
        
        -- Resultado del partido
        t.goals_for,
        opp.goals_for AS goals_against,
        CASE 
            WHEN t.goals_for > opp.goals_for THEN 'W'
            WHEN t.goals_for < opp.goals_for THEN 'L'
            ELSE 'D'
        END AS match_result,
        
        -- Métricas esperadas (Generadas vs Concedidas)
        t.xg_for,
        opp.xg_for AS xg_against,
        ROUND(t.xg_for - opp.xg_for, 4) AS xg_diff,
        t.xgot_for,
        opp.xgot_for AS xgot_against,
        t.shots_for,
        opp.shots_for AS shots_against,
        t.sav_total,
        
        -- Dominio y territorio
        t.xt_total,
        t.xt_passes,
        t.xt_carries,
        opp.xt_total AS xt_conceded,
        
        -- Circulación y estimación de posesión (% de pases totales del partido)
        t.passes_total,
        t.passes_completed,
        ROUND((t.passes_completed * 100.0) / NULLIF(t.passes_total, 0), 1) AS pass_accuracy_pct,
        ROUND((t.passes_completed * 100.0) / NULLIF(t.passes_completed + opp.passes_completed, 0), 1) AS possession_proxy_pct,
        
        -- Progresión y último tercio
        t.prog_passes_total,
        t.prog_carries_total,
        t.crosses_total,
        t.crosses_completed,
        t.key_passes_total,
        t.assists_total,
        t.dribbles_total,
        t.dribbles_completed,
        
        -- Presión y contención
        t.recoveries_total,
        t.interceptions_total,
        t.tackles_total,
        t.tackles_won_total,
        t.clearances_total,
        t.blocks_total

    FROM team_raw t
    LEFT JOIN team_raw opp 
        ON t.match_id = opp.match_id AND t.team_name != opp.team_name
    ORDER BY t.round, t.match_id, t.team_side DESC
) TO '../data/gold/team_match_gold.parquet' (FORMAT PARQUET);
"""

con.execute(query_team_match_gold)

con.execute("""
    SELECT 
        round,
        team_name,
        opponent_name,
        match_result,
        goals_for || ' - ' || goals_against AS score,
        xg_for,
        xg_against,
        xg_diff,
        possession_proxy_pct,
        shots_for,
        shots_against
    FROM '../data/gold/team_match_gold.parquet'
    WHERE team_name = 'Koninklijke Sporting Hasselt'
    ORDER BY round
""").df()

,round,team_name,opponent_name,match_result,score,xg_for,xg_against,xg_diff,possession_proxy_pct,shots_for,shots_against
0,1,Koninklijke Sporting Hasselt,FCV Dender,W,1.0 - 0.0,1.9274,1.9380,-0.0106,38.5,13.0,27.0
1,2,Koninklijke Sporting Hasselt,Royal Francs Borains,W,1.0 - 0.0,1.1815,0.9282,0.2533,60.7,11.0,12.0
2,3,Koninklijke Sporting Hasselt,KAA Gent Reserve U23,L,0.0 - 2.0,2.3587,1.4777,0.8810,55.8,20.0,8.0
